# 06 — Blocking

Without blocking, matcher compares every left row to every right row (or within-block in dedup). That's fine for tiny tables but blows up on larger data. **Blocking** restricts comparisons to rows that share the same value of a **blocking key** (e.g. `zip_code`): only rows in the same block are compared. You get fewer comparisons (faster) and often fewer spurious pairs—but if a true pair has *different* key values, they'll never be compared and recall drops. Here we use the **blocking_evaluation** set: 30 known pairs where 15 share the same zip and 15 have *different* zips (same person, different zip on left vs right). So blocking on zip finds only 15—you'll see recall drop and learn why the choice of key matters.

**Back:** [05 Design Your Algorithm](05_design_algorithm.ipynb) · **Next:** [07 Deduplication](07_deduplication.ipynb)

## 1. Load data and ground truth

We use the **blocking_evaluation** set (50 left, 50 right, 30 known pairs): 15 pairs share the same zip; 15 pairs are the same person but have different zip codes on left vs right, so blocking on zip puts them in different blocks. Ground truth is eval_left_1..30 / eval_right_1..30.

In [ ]:
import sys
from pathlib import Path
import polars as pl
from matcher import Matcher

_tutorial = Path.cwd() / "docs" / "tutorial"
if not (_tutorial / "tutorial_data").exists():
    _tutorial = (
        Path.cwd()
        if (Path.cwd() / "tutorial_data").exists()
        else Path.cwd().parent.parent / "docs" / "tutorial"
    )
sys.path.insert(0, str(_tutorial))
from tutorial_data import load_blocking_evaluation

left, right, ground_truth = load_blocking_evaluation()
matcher = Matcher(left=left, right=right, left_id="id", right_id="id")
print(f"Left: {left.shape[0]} rows, Right: {right.shape[0]} rows, Ground truth: {ground_truth.shape[0]} pairs")

## 2. What blocking does

Matcher's `blocking_key` must be a column that exists in both tables with the same name. Rows are grouped by that column's value; comparisons happen only *within* each group. So if you block on `zip_code`, a left row in zip 10001 is only compared to right rows in 10001—never to 10002. That cuts the number of comparisons and can reduce false positives, but any true pair that has different zip codes will never be compared, so you'll miss them (recall drops).

## 3. With vs without blocking

Run the same rules (e.g. email) with and without a blocking key. Always evaluate on the same ground truth. If recall is lower with blocking, your key is splitting true pairs—try a different key or omit blocking.

In [ ]:
results_no_block = matcher.match(rules="email")
results_with_block = matcher.match(rules="email", blocking_key="zip_code")

m_no = results_no_block.evaluate(ground_truth, left_id_col="id", right_id_col="id_right")
m_block = results_with_block.evaluate(ground_truth, left_id_col="id", right_id_col="id_right")

print("No blocking:    ", f"recall={m_no['recall']:.2%}, matches={results_no_block.count}")
print("With zip_code: ", f"recall={m_block['recall']:.2%}, matches={results_with_block.count}")

On this sample, 15 true pairs have *different* zips (left in one block, right in another), so blocking by `zip_code` puts them in different blocks and recall drops to 50%. That's the signal: your blocking key is splitting true pairs. Choose a key that true pairs are likely to share (e.g. area code, first letter of surname) or run without blocking if no key is safe.

## 4. Blocking with fuzzy

You can pass `blocking_key` to `match_fuzzy()` as well. Same idea: fuzzy comparisons happen only within blocks. So blocking applies to both exact and fuzzy matching.

In [ ]:
# Example: fuzzy on full_name, only within zip_code blocks
left = left.with_columns(
    (pl.col("first_name").fill_null("") + " " + pl.col("last_name").fill_null("")).str.strip_chars().alias("full_name")
)
right = right.with_columns(
    (pl.col("first_name").fill_null("") + " " + pl.col("last_name").fill_null("")).str.strip_chars().alias("full_name")
)
matcher_block = Matcher(left=left, right=right, left_id="id", right_id="id")
fuzzy_block = matcher_block.match_fuzzy(field="full_name", threshold=0.85, blocking_key="zip_code")
print(f"Fuzzy matches with zip_code blocking: {fuzzy_block.count}")

You now have **blocking** (06). Next: [07 Deduplication](07_deduplication.ipynb) — same ideas on a single table with `Deduplicator`.